# Lesson 03: Cases, debt and equity

> **Question.** How does the base plan perform in weak, reference and strong markets,
> with and without debt? Can we pay the bills when they fall due?
>
> **You will learn to:**
>
> 1. build dated cash flows from a base plan, and tell the base plan from the base case;
> 2. declare named market cases and solve them in one batch;
> 3. fund the plan with equity first, then a loan, and report unlevered NPV, equity
>    NPV and equity IRR for each case and equity level;
> 4. find the land price that gives a target equity IRR;
> 5. test construction funding: the loan principal, what is left to spend and the
>    shortfall.
>
> **Before you start.** Lessons 01 and 02.
>
> **Time.** 60 minutes core, 25 minutes optional (more equity levels, the Run as
> evidence).
>
> **Not in this lesson.** Many market futures (lesson 04). Changing the plan's dates
> by rule (lesson 05).
>
> **Source.** Table 4.1 in de Neufville and Geltner (2018), used with the
> publisher's permission and with attribution. The numbers are the property's own.

## 1. The base plan

The base plan says what is built and when. Shared works in 2027, Building
A in 2028–2029 and Building B in 2030–2031. Each building opens the next January.
The base case is the financial result of that plan under stated reference
assumptions. The plan does not change in this lesson; the assumptions do.

In [ ]:
from datetime import date

import helpers

import rangekeeper as rk

In [ ]:
plan = helpers.load_base_plan()
plan  # dates, budgets, incomes and funding, with origins

In [ ]:
yearly = rk.duration.Frequency.YEAR
years = rk.duration.Span(start=date(2027, 1, 1), end=date(2038, 1, 1)).periods(
    frequency=yearly
)
helpers.show_timeline(of=plan)

> **Finance note.** A development budget has hard costs (construction), soft costs
> (fees and approvals) and a contingency. Land is a separate line. The plan's budgets
> leave out the data package's 4 million AUD allowance for construction finance:
> section 6 calculates the interest on the loan instead.

## 2. The budget, by hand once

Building A's budget is its area by use times the construction rate, plus
10 % soft costs and 10 % contingency. Calculate it once by hand and compare it with
the plan's total. The other packages follow the same rule.

In [ ]:
areas = plan.areas["building_a"]
hard_costs = rk.Quantity(text="0 AUD")
for use in areas:
    hard_costs = hard_costs + areas[use] * plan.rates[use]
soft_costs = 0.10 * hard_costs
contingency = 0.10 * hard_costs
budget = hard_costs + soft_costs + contingency
assert rk.quantity.isclose(actual=budget, expected=plan["a_cost"])

## 3. The dated Model

As in lesson 02, start an empty Model and edit it. Declare the inputs
with their units. Dates are values too, so a later lesson can change them. The
plan's sequence relates them: the shared works start on the start date, and the
shared plinth opens a year later; Building A follows a year later, each building
takes two years to build, and Building B cannot start before the shared works
finish. Group equations in named formulations, like
the sections of a pro forma: `add_formulation(name=)` makes one, and its
`equations(...)` adds to it. A formulation changes nothing in the solve; it organises
the display and the trace. The date equations form "Dates".

In [ ]:
model = rk.Model(name="Mixed-use base plan")
draft = model.edit()
project = draft.add_entity(code="project", classification="project")
project.date(key="base_date")
project.date(key="sale_date")
project.date(key="start_date")
project.date(key="b_start")
dates = project.add_formulation(name="Dates")
dates.equations(
    "shared_start = start_date",
    "shared_open = shared_start + 1 year",
    "a_start = shared_start + 1 year",
    "a_open = a_start + 2 years",
    "b_open = b_start + 2 years",
    "b_start >= shared_start + 1 year",
)

The amounts and rates. Incomes and operating costs are annual amounts
at 2027 prices; budgets and renewals are totals. The shared plinth has its own
operating cost and renewal: it serves both buildings, so the plan does not split
them between the buildings. The cases in section 5 change two of the rates.

In [ ]:
project.value(key="land_cost", units="AUD")
project.value(key="shared_cost", units="AUD")
project.value(key="a_cost", units="AUD")
project.value(key="b_cost", units="AUD")
project.value(key="a_income", units="AUD pa")
project.value(key="b_income", units="AUD pa")
project.value(key="shared_opex", units="AUD pa")
project.value(key="a_opex", units="AUD pa")
project.value(key="b_opex", units="AUD pa")
project.value(key="shared_renewal", units="AUD")
project.value(key="a_renewal", units="AUD")
project.value(key="b_renewal", units="AUD")
project.value(key="rent_growth", units="% pa")
project.value(key="opex_growth", units="% pa")
project.value(key="exit_cap_rate", units="ratio")
project.value(key="discount_rate", units="% pa")

Some Flows are made from amounts, by a profile or an event. Nothing in
their equations says which Periods they cover, so declare them here, each with its
Periods. Every other value takes its units and Periods from its equations: a sum of
yearly Flows is yearly, and `commit()` names any value that it cannot work out.
Declare every input, and every result that you will rely on. A declaration is a
check: `commit()` fails if an equation gives the value other units.

In [ ]:
project.flow(key="land", units="AUD", periods=years)
project.flow(key="shared_works", units="AUD", periods=years)
project.flow(key="a_works", units="AUD", periods=years)
project.flow(key="b_works", units="AUD", periods=years)
project.flow(key="a_rent", units="AUD", periods=years)
project.flow(key="b_rent", units="AUD", periods=years)
project.flow(key="shared_costs", units="AUD", periods=years)
project.flow(key="a_costs", units="AUD", periods=years)
project.flow(key="b_costs", units="AUD", periods=years)
project.flow(key="shared_renewals", units="AUD", periods=years)
project.flow(key="a_renewals", units="AUD", periods=years)
project.flow(key="b_renewals", units="AUD", periods=years)
project.flow(key="sale", units="AUD", periods=years)

Development, the next group. The site is bought on the start date.
Development payments are negative; `rx.profile.uniform` divides an amount evenly
over the years from a start date. In lesson 01, `rk.` functions calculated a result
at once. In an equation, functions start with `rx.`: they write a relation that the
solve uses.

In [ ]:
development = project.add_formulation(name="Development")
development.equations(
    "land = -rx.flow.event(amount=land_cost, on=start_date)",
    "shared_works = -rx.profile.uniform(total=shared_cost, start=shared_start, length=1 year)",
    "a_works = -rx.profile.uniform(total=a_cost, start=a_start, length=2 years)",
    "b_works = -rx.profile.uniform(total=b_cost, start=b_start, length=2 years)",
    "works = shared_works + a_works + b_works",
)

Operations. Rents grow on one calendar from 2027, but each building
earns nothing until it opens. The plinth's operating cost starts when it opens, in
2028. Renewals come every five years after opening.

In [ ]:
operations = project.add_formulation(name="Operations")
operations.equations(
    "a_rent = rx.profile.compounding(initial=a_income, rate=rent_growth, start=a_open)",
    "b_rent = rx.profile.compounding(initial=b_income, rate=rent_growth, start=b_open)",
    (
        "shared_costs = rx.profile.compounding(initial=shared_opex, rate=opex_growth,"
        " start=shared_open)"
    ),
    "a_costs = rx.profile.compounding(initial=a_opex, rate=opex_growth, start=a_open)",
    "b_costs = rx.profile.compounding(initial=b_opex, rate=opex_growth, start=b_open)",
    "income = a_rent + b_rent",
    "opex = -(shared_costs + a_costs + b_costs)",
)

Renewals come every five years after each package opens, at costs that
grow from 2027.

In [ ]:
operations.equations(
    (
        "shared_renewals = rx.profile.compounding(initial=shared_renewal,"
        " rate=opex_growth, start=shared_open + 5 years, every=5 years)"
    ),
    (
        "a_renewals = rx.profile.compounding(initial=a_renewal, rate=opex_growth,"
        " start=a_open + 5 years, every=5 years)"
    ),
    (
        "b_renewals = rx.profile.compounding(initial=b_renewal, rate=opex_growth,"
        " start=b_open + 5 years, every=5 years)"
    ),
    "renewals = -(shared_renewals + a_renewals + b_renewals)",
    "operating = income + opex + renewals",
)

The sale and the value. These are lesson 01's ideas, now as equations.

In [ ]:
project.value(key="unlevered_npv", units="AUD")  # a result we rely on: checked
sale_and_value = project.add_formulation(name="Sale and value")
sale_and_value.equations(
    "exit_value = operating.at(on=sale_date + 1 year) / exit_cap_rate",
    "sale = rx.flow.event(amount=exit_value, on=sale_date)",
    "project_cash = land + works + operating + sale",
    "net_cash_flow = project_cash.until(on=sale_date)",
    "unlevered_npv = rx.finance.npv(flow=net_cash_flow, rate=discount_rate, on=base_date)",
)
model = draft.commit()
model.entity(code="project").values  # units of each value: declared or inferred

## 4. The base case

**Predict:** is the unlevered NPV positive at the reference assumptions?
Fix the plan's inputs and the reference assumptions. Everything else is free.

In [ ]:
base = (
    model.specify(name="Base case")
    .fix(base_date=plan["base_date"], sale_date=plan["sale_date"])
    .fix(start_date=plan["start_date"], b_start=plan["b_start"])
    .fix(land_cost=plan["land_cost"], shared_cost=plan["shared_cost"])
    .fix(a_cost=plan["a_cost"], b_cost=plan["b_cost"])
    .fix(a_income=plan["a_income"], b_income=plan["b_income"])
    .fix(shared_opex=plan["shared_opex"], a_opex=plan["a_opex"])
    .fix(b_opex=plan["b_opex"], shared_renewal=plan["shared_renewal"])
    .fix(a_renewal=plan["a_renewal"], b_renewal=plan["b_renewal"])
)
base = base.fix(rent_growth="3 % pa", exit_cap_rate="5 %")
base = base.fix(discount_rate="7 % pa", opex_growth="3.5 % pa")
outcome = rk.execution.solve(specification=base)
outcome["unlevered_npv"]

In [ ]:
outcome.stream("land", "works", "income", "opex", "renewals", "sale").display()

In [ ]:
helpers.plot_cash_flows(outcome=outcome, key="net_cash_flow")

## 5. Named cases

A case is one named set of assumptions. A batch Specification holds the
cases; one solve gives one comparable table. Every case uses the same plan and the
same valuation conventions.

In [ ]:
cases = {
    "weak": {"rent_growth": "1 % pa", "exit_cap_rate": "6 %"},
    "reference": {"rent_growth": "3 % pa", "exit_cap_rate": "5 %"},
    "strong": {"rent_growth": "5 % pa", "exit_cap_rate": "4.5 %"},
}
cases_outcome = rk.execution.solve(specification=base.cases(cases=cases))
cases_outcome.table("unlevered_npv")

> **Under the hood.** `base.cases(...)` makes one batch Specification. Each case
> overrides the base's fixed values by key. The solve builds the shared linear
> structure once and solves each case.

## 6. Debt and equity: equity first, then the loan

The owner commits 50 million AUD of equity and pays it in as costs fall
due. Lenders usually require the equity to be spent first. When the equity is used
up, a construction loan pays the rest, up to its limit. Income and the sale repay the
loan first; the rest goes to equity. The funding need is the running total that the
project has not paid from its own cash; the loan principal is the part of it that the
equity cannot cover. `diff()` gives each period's change in a running total: the
reverse of `cumulative()`. Add the equations to a new revision of the Model.

In [ ]:
draft = model.edit()
project = draft.entity(code="project")
project.value(key="equity", units="AUD")
project.value(key="loan_limit", units="AUD")
project.value(key="loan_rate", units="% pa")
project.value(key="equity_rate", units="% pa")
project.value(key="equity_npv", units="AUD")  # results we rely on: checked
project.value(key="equity_irr", units="% pa")
funding = project.add_formulation(name="Funding")
funding.equations(
    "funding_need = -(net_cash_flow + loan_interest).cumulative()",
    "loan_principal = rx.max(0 AUD, funding_need - equity)",
    "loan_draws = loan_principal.diff()",
    "loan_interest = -rx.finance.interest(balance=loan_principal, rate=loan_rate)",
)

Equity receives whatever the project pays that does not go to or come from
the loan. Its cash flow is negative while it pays costs, zero while the loan pays,
and positive once the loan is repaid.

In [ ]:
funding.equations(
    "equity_cash_flow = net_cash_flow + loan_interest + loan_draws",
    "equity_npv = rx.finance.npv(flow=equity_cash_flow, rate=equity_rate, on=base_date)",
    "equity_irr = rx.finance.irr(flow=equity_cash_flow)",
)
model = draft.commit()

In [ ]:
levered = base.on(model=model).fix(
    equity=plan["equity"],
    loan_limit=plan["loan_limit"],
    loan_rate="6 % pa",
    equity_rate="12 % pa",
)
levered_outcome = rk.execution.solve(specification=levered.cases(cases=cases))
levered_outcome.table("unlevered_npv", "equity_npv", "equity_irr")

> **Finance note.** Debt makes the equity's results larger in both directions. When
> the property earns more than the loan rate, debt raises the equity IRR. IRR has
> failure cases: a cash flow with more than one change of sign can have more than one
> IRR, and the solve then reports `irr_not_unique`.

## 7. Equity levels

**Predict:** in the reference case, does the equity NPV rise when the
owner commits less equity and borrows more? Compare three levels in one batch.

In [ ]:
levels = {
    "80 m equity": {"equity": "80e6 AUD"},
    "50 m equity": {"equity": "50e6 AUD"},
    "30 m equity": {"equity": "30e6 AUD"},
}
rk.execution.solve(specification=levered.cases(cases=levels)).table(
    "equity_npv", "equity_irr", "loan_principal"
)

## 8. A target equity IRR

The back door again, now on dated cash flows. The investor needs a 15 %
equity IRR. What land price gives it? Fix the IRR and free the land cost. With the
IRR fixed, the IRR equation becomes "NPV at 15 % is zero", which is linear.
The cascade's `max` has two sides; the solve finds which side applies in each period
and checks it after the solve.

In [ ]:
target = levered.fix(equity_irr="15 % pa").free("land_cost")
target_outcome = rk.execution.solve(specification=target)
target_outcome["land_cost"]

## 9. Can we pay the bills? By hand once

A positive NPV does not pay a construction invoice. Payments come at the
end of each year. Split them by hand once, through 2028, the first year in which
their running total passes the equity: equity pays first, and the loan pays what the
equity cannot. The need is the running total of the net cash flow, so it includes
the plinth's operating cost. The loan also pays its own interest for 2028, so it
lends the rest of the need divided by one less a year's interest on each AUD.

In [ ]:
need = -outcome["net_cash_flow"].cumulative().at(on=date(2028, 1, 1))  # to end 2028
from_equity = min(need, plan["equity"])
loan_rate = levered_outcome.case(key="reference")["loan_rate"]
year_interest = (loan_rate * rk.Quantity(text="1 year")).number  # per AUD of loan
from_loan = (need - from_equity) / (1 - year_interest)
from_equity, from_loan

The Model makes the same split every year. Read the reference case from
the batch. This is the spreadsheet corkscrew: the running total of the need, and the
loan's part of it.

In [ ]:
reference_outcome = levered_outcome.case(key="reference")
principal_2028 = reference_outcome["loan_principal"].at(on=date(2028, 1, 1))
assert rk.quantity.isclose(actual=principal_2028, expected=from_loan)
reference_outcome.stream(
    "net_cash_flow", "funding_need", "loan_draws", "loan_principal", "equity_cash_flow"
)

## 10. What is left to spend

What can still be spent is the available equity and the available loan.
`project.formulation(name="Funding")` finds the formulation that cell 12 made.
`loan_available` is what can still be drawn under the limit; `loan_principal` is what
is owed. A shortfall is loan principal above the limit. Building A's income helps to
pay for Building B, so the need includes the operating cash flow.

In [ ]:
draft = model.edit()
project = draft.entity(code="project")
project.value(key="peak_shortfall", units="AUD")  # a result we rely on: checked
funding = project.formulation(name="Funding")
funding.equations(
    "equity_available = rx.max(0 AUD, equity - funding_need)",
    "loan_available = loan_limit - loan_principal",
    "shortfall = rx.max(0 AUD, -loan_available)",
    "peak_shortfall = shortfall.max()",
)
model = draft.commit()

In [ ]:
funded = levered.on(model=model)
funded_outcome = rk.execution.solve(specification=funded.cases(cases=cases))
funded_outcome.table("unlevered_npv", "equity_npv", "equity_irr", "peak_shortfall")

In [ ]:
helpers.plot_flows(
    outcome=funded_outcome.case(key="weak"), keys=["equity_available", "loan_available"]
)

> **Finance note.** The shortfall uses only the equity and the loan that exist at each
> date. It assumes no future sale or refinance.

## 11. Optional: commitments

A package's remaining payments become committed when it starts. What is
still to be paid is the running total less the whole total; `since` makes it zero
before the start date. The commitment gap is the largest amount by which committed
but unpaid payments exceed the available equity and the available loan. Lesson 05
uses it to check whether a start can be funded.

In [ ]:
draft = model.edit()
project = draft.entity(code="project")
funding = project.formulation(name="Funding")
funding.equations(
    "shared_unpaid = (shared_works.cumulative() - shared_works.total()).since(on=shared_start)",
    "a_unpaid = (a_works.cumulative() - a_works.total()).since(on=a_start)",
    "b_unpaid = (b_works.cumulative() - b_works.total()).since(on=b_start)",
    "committed = shared_unpaid + a_unpaid + b_unpaid",
    "commitment_gap = rx.max(0 AUD, (committed - equity_available - loan_available).max())",
)
model = draft.commit()
committed_outcome = rk.execution.solve(
    specification=funded.on(model=model).cases(cases=cases)
)
committed_outcome.table("peak_shortfall", "commitment_gap")

> **Finance note.** When Building B starts, its whole budget is committed, even though
> the payments come over two years.

## 12. Optional, under the hood: the Run as evidence

`solve` keeps everything in memory. `execute` also publishes the outputs
and a Run to a store. A Run records the Specification, the Model revision, the
solver, the status and every case. Replay reads it in a fresh kernel and checks
that the result is the same. A namespace gives the same IDs on every build.

In [ ]:
store = rk.io.DirectoryStore(path="runs/03")
executed = rk.execution.execute(specification=funded.cases(cases=cases), store=store)
executed.run.id

This cell needs only the folder of the store. To see that, restart the
kernel, import `rangekeeper as rk` and run it.

In [ ]:
store = rk.io.DirectoryStore(path="runs/03")
run = store.find_runs(specification="Base case")[-1]
rk.execution.replay(run=run, store=store).raise_for_status()

> **Under the hood: the same IDs on every build.** `rk.Model(name=, namespace=)` takes
> a namespace, a UUID. The IDs that a draft makes then come from it, so the same inputs
> give the same IDs. The tested contract and `rk.model.namespace(name=)` follow in
> [#147](https://github.com/daniel-fink/rangekeeper/issues/147).

## Exercises

**Try it 1: a stress case.** Solve the funded plan with no rent growth and a 7 % exit
cap rate. Compare it with the reference case. Does the peak shortfall change? Which
cash flow causes the change?

In [ ]:
stress_outcome = rk.execution.solve(
    specification=funded.fix(rent_growth="0 % pa", exit_cap_rate="7 %")
)
rk.execution.compare(
    outcomes={
        "Reference": funded_outcome.case(key="reference"),
        "Stress": stress_outcome,
    },
    keys=["peak_shortfall", "equity_npv"],
    baseline="Reference",
)
# The shortfall increases. Building A's income is lower, so it pays less of Building
# B's works. The exit cap rate changes the equity NPV but not the shortfall, because
# the sale occurs after all works are paid.

**Try it 2: the target IRR in the weak case.** What land price gives a 15 % equity
IRR in the weak market?

In [ ]:
weak_target_outcome = rk.execution.solve(
    specification=target.fix(rent_growth="1 % pa", exit_cap_rate="6 %")
)
assert weak_target_outcome["land_cost"] < target_outcome["land_cost"]

**Try it 3: start Building B a year later.** In which year is the available loan
smallest now?

In [ ]:
later_outcome = rk.execution.solve(specification=funded.fix(b_start=date(2031, 1, 1)))
available = later_outcome["loan_available"]
smallest = available.min()
for movement in available.movements:
    if movement.number == smallest.number:
        print(movement.period, smallest)

## Checks and recap

The checks confirm the lesson's results.

In [ ]:
npvs = cases_outcome.table("unlevered_npv").column("unlevered_npv")
assert list(npvs) == sorted(npvs)  # weak < reference < strong
cash = target_outcome["equity_cash_flow"]
target_npv = rk.finance.npv(flow=cash, rate="15 % pa", on=date(2027, 1, 1))
assert rk.quantity.isclose(actual=target_npv, expected="0 AUD", tolerance="1 AUD")
reference_outcome = funded_outcome.case(key="reference")
principal_2028 = reference_outcome["loan_principal"].at(on=date(2028, 1, 1))
assert rk.quantity.isclose(actual=principal_2028, expected=from_loan)

**Recap.**

1. The base plan is the dates; the base case is its result under reference
   assumptions. Cases change assumptions, not the plan.
2. One batch Specification gives one comparable table: unlevered and levered results,
   and the funding test, for every case.
3. A fixed equity IRR is a question like any other: the back door works on dated cash
   flows too.

**Next.** Lesson 04 replaces three named cases with many generated market paths.